# 3-stage learned-router pipeline

This notebook trains the official three-stage predicted-router system at five training-noise levels. Raw runs are stored under `output/stage_three/runs/`; grouped research artifacts are stored under `output/stage_three/`, and curated copies are published under `expected_output/stage_three/`. The specialist decision is made by a learned router from predicted coefficients; true shape labels are used only as training labels and diagnostics.

## Stage contracts

## Architecture

The three-stage architecture is:

22 gradient features -> Stage 1 coefficient regressor -> predicted coefficients -> predicted router -> general mask head or two-circle specialist mask head.

Stage 1 predicts coefficient features. The general and specialist heads reconstruct masks from those predicted coefficients. The router selects the specialist using predicted coefficient features only; it never receives the true test shape.

The clean test result is official. Test-noise curves are supplementary robustness measurements. N affects the coefficient representation, so errors in Stage 1 can influence both mask heads and the router.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config.official import OFFICIAL_TRAIN_SIGMAS, official_three_stage_config
from workflows.task9.datasets import build_task9_general_dataset
from experiments import (layer_table, parameter_count, publish_expected_output, publish_run_artifacts)
from workflows.three_models import run_three_models_with_predictor
from models import Stage1Regressor
from models.three_models import CoefficientToGeneralMaskModel, CoefficientToSpecialistMaskModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT = ROOT / 'output' / 'stage_three'
TRAIN_SIGMAS = OFFICIAL_TRAIN_SIGMAS
# This workflow must train to return a live predictor for the noise sweep.

def make_config(train_sigma):
    return official_three_stage_config(train_sigma, output_root=ROOT / 'output')

stage1_check = Stage1Regressor(input_dim=22, output_dim=22, hidden_dims=(256, 512, 256), dropout_rates=(0.2, 0.2, 0.2))
general_check = CoefficientToGeneralMaskModel(input_dim=22, output_dim=32 * 32, hidden_dims=(512, 1024, 2048), dropout_rates=(0.2, 0.2, 0.2))
specialist_check = CoefficientToSpecialistMaskModel(input_dim=22, output_dim=32 * 32, hidden_dims=(1024, 2048, 4096), dropout_rates=(0.3, 0.3, 0.3))
for name, model, dummy in (('stage1', stage1_check, torch.zeros(2, 22)), ('general_head', general_check, torch.zeros(2, 22)), ('specialist_head', specialist_check, torch.zeros(2, 22))):
    print(name, parameter_count(model))
    layer_table(model, dummy)

stage1 {'total': 274454, 'trainable': 274454}
general_head {'total': 4734464, 'trainable': 4734464}
specialist_head {'total': 14710784, 'trainable': 14710784}


In [ ]:
comparison_rows = []
for train_sigma in TRAIN_SIGMAS:
    config = make_config(train_sigma)
    print(f'\n=== training sigma={train_sigma:g} ===')
    summary, predict_logits = run_three_models_with_predictor(config, device=DEVICE)
    bundle = build_task9_general_dataset(config)
    threshold = float(summary['threshold_summary']['selected_threshold'])
    publish_run_artifacts('stage_three', config.run_output_dir, train_sigma, grouped_root=ROOT / 'output')
    clean = summary['metrics']['stage2_test']
    comparison_rows.append({'train_sigma': train_sigma, 'clean_test_iou': clean['mean_iou'], 'clean_pixel_accuracy': clean['pixel_accuracy'], 'threshold': threshold})
comparison = pd.DataFrame(comparison_rows)
publish_expected_output('stage_three', output_root=ROOT / 'output')
display(comparison)


=== training sigma=0 ===
Epoch 1/200 | step 10 | train_loss=0.712598 | val_loss=1.089117
Epoch 1/200 | step 20 | train_loss=0.977626 | val_loss=1.053261
Epoch 1/200 | step 30 | train_loss=0.878539 | val_loss=1.013372
Epoch 1/200 | step 40 | train_loss=0.837027 | val_loss=0.964840
Epoch 1/200 | step 50 | train_loss=0.785617 | val_loss=0.915913
Epoch 1/200 | step 60 | train_loss=0.824423 | val_loss=0.861639
Epoch 1/200 | step 70 | train_loss=0.796038 | val_loss=0.818420
Epoch 1/200 | step 80 | train_loss=0.761916 | val_loss=0.793303
Epoch 1/200 | step 90 | train_loss=0.745099 | val_loss=0.768253
Epoch 1/200 | step 100 | train_loss=0.724882 | val_loss=0.748462
Epoch 1/200 | step 110 | train_loss=0.716659 | val_loss=0.728622
Epoch 1/200 | step 120 | train_loss=0.707742 | val_loss=0.699459
Epoch 1/200 | step 130 | train_loss=0.694863 | val_loss=0.671819
Epoch 1/200 | step 140 | train_loss=0.703896 | val_loss=0.633429
Epoch 1/200 | step 150 | train_loss=0.695684 | val_loss=0.600175
Epoch 1/

The router is trained using Stage 1 predicted coefficients and the known training shape labels. Those labels are not passed to the router during validation or test inference. Official metrics use clean validation and test measurements.